# R7.5 · Multi-year expanding-window rolling hindcasts
### **hybrid deep-learning prediction** — frozen single-model architecture

**独立 Notebook · RTX 5080 / WSL / VS Code**，无外网要求。不调用 ADT 进行训练、模型选择或早停。架构等价于 R6.5 的 `CrossDrop60`（内部可追溯名），论文和图表统一显示 **hybrid deep-learning prediction**。

核心合同：**180 日输入 → +30/+60/+90 日目标**；GroupNorm(8)、PCSkip 双分支、训练时 `p=0.60` 交叉交换 Dropout、推理时关闭、物理距离辅助损失 `alpha=0.40`。每个测试年份独立拟合仅训练期 EOF(99%)、训练模型、验证早停、整年（有 90 日共同目标的）滚动起报。

严格区别：*模型结构与超参数冻结* ≠ *同一个历史权重可对所有年份声称后报*。旧 2014/2019 检查点仅在原合同精确吻合时复用；新年份训练历史专属权重。2014/2019 已用于研发，后报全部年份均不称为预注册独立盲测。下轮同规模 LSTM 等基线需继承这里输出的 `folds` 和 `issue_date`。

**默认** 10 个预设年度折次：2005、2008、2011、2014、2017、2019、2021、2022、2023、2024。可以设置 `ALL_ANNUAL=True` 扩展为每年 2005–2024（20 折）。不是事后挑选性能最好的年份。每折 seeds=(71,72)。


In [ ]:
# 0｜配置与文件系统保护（不改变 R6.5 网络结构）
from pathlib import Path
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
import os, sys, json, hashlib, math, time, random, gc, traceback, zipfile, warnings, re, copy
import numpy as np, pandas as pd
from scipy.spatial import cKDTree
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from IPython.display import display, FileLink
from tqdm.auto import tqdm

PAPER_NAME = 'hybrid deep-learning prediction'
INTERNAL_ARCH = 'R65-CrossDrop60'
@dataclass
class Settings:
    ROOT: str = '/mnt/d/KuroshioDaily1993_2025'
    DATASET_REL: str = 'final/dataset/daily_dual_branch_paths_1993_2025_FINAL.npz'
    OUTPUT_REL: str = 'R7_5_HybridDeepLearning_MultiYearRollingHindcast'
    SOURCE_R65_WEIGHTS_REL: str = 'R6_5_跨分支交换随机屏蔽实验/模型权重/R65_Checkpoints'
    HISTORY: int = 180
    LEADS: tuple = (30,60,90)
    TEST_YEARS: tuple = (2005,2008,2011,2014,2017,2019,2021,2022,2023,2024)
    ALL_ANNUAL: bool = False
    SEEDS: tuple = (71,72)
    PURGE_DAYS: int = 7
    MAX_INPUT_GAP: int = 2
    MAX_MISSING_FRAC: float = .03
    DROP_BRANCH1_FIRST: int = 12
    BRANCH2_MAX_LON: float = 145.0
    EOF_VARIANCE_TARGET: float = .99
    BATCH_SIZE: int = 32
    DROPOUT: float = .30
    CROSS_DROPOUT: float = .60
    ALPHA: float = .40
    KM_SCALE: float = 25.0
    HUBER_BETA: float = .50
    LR: float = .001
    WEIGHT_DECAY: float = .0001
    EPOCHS: int = 36
    PATIENCE: int = 8
    USE_AMP: bool = True
    FORCE_TRAIN: bool = False
    USE_EXISTING_R65_WEIGHTS: bool = True
    RUN_FULL: bool = True
    ONLY_PREFLIGHT: bool = False
    MIN_TRAIN: int = 500
    MIN_VAL: int = 100
    MIN_TEST: int = 100
    NUM_WORKERS: int = 0
    BOOT_REPS: int = 400
    BOOT_BLOCK_DAYS: int = 30
    INCLUDE_PREDICTIONS_IN_ZIP: bool = False
    INCLUDE_WEIGHTS_IN_ZIP: bool = False
    ZIP_PREDICTION_EXAMPLES: int = 6
    FIG_DPI: int = 240

c=Settings()
ROOT=Path(c.ROOT); DS=ROOT/c.DATASET_REL
RUN=ROOT/c.OUTPUT_REL
TABLES=RUN/'tables';WEIGHTS=RUN/'weights';FORECAST=RUN/'forecast_arrays';LOGS=RUN/'logs';FIG=RUN/'figures_en';REPORTS=RUN/'reports';EOFSTORE=RUN/'trainonly_eof'
for p in [RUN,TABLES,WEIGHTS,FORECAST,LOGS,FIG,REPORTS,EOFSTORE]:p.mkdir(parents=True,exist_ok=True)
ARCHIVE=ROOT/'R75_上传给ChatGPT_多年滚动后报_hybrid_deep-learning_prediction.zip'
DEV=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if not DS.is_file():raise FileNotFoundError('冻结数据集不存在: '+str(DS))
if not torch.cuda.is_available():warnings.warn('当前不是 CUDA，模型可以运行但速度会较慢；建议RTX5080 WSL')
if c.ALL_ANNUAL:c.TEST_YEARS=tuple(range(2005,2025))
if len(c.TEST_YEARS)!=len(set(c.TEST_YEARS)):raise RuntimeError('测试年份重复')

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(8<<20),b''):h.update(b)
    return h.hexdigest()
DATASET_SHA=sha256_file(DS)
print('Device:',DEV,'torch:',torch.__version__,'dataset SHA256:',DATASET_SHA)
print('Publication name:',PAPER_NAME,'folds:',c.TEST_YEARS,'seeds:',c.SEEDS)
(REPORTS/'R75_冻结实验合同.json').write_text(json.dumps({'paper_model_name':PAPER_NAME,'internal_arch':INTERNAL_ARCH,'dataset_sha256':DATASET_SHA,'settings':asdict(c),'not_blind':True},ensure_ascii=False,indent=2),encoding='utf-8')


In [ ]:
# 1｜按冻结数据格式恢复双分支/日历，不修改样本或标签
with np.load(DS,allow_pickle=False) as z:
    dates=pd.DatetimeIndex(pd.to_datetime(z['dates'])).normalize()
    b1=np.asarray(z['branch1_lon'],np.float32)
    lat1=np.asarray(z['branch1_target_lats'],float)
    b2=np.asarray(z['branch2_lat'],np.float32)
    lon2=np.asarray(z['branch2_target_lons'],float)
if dates.has_duplicates or not(len(dates)==len(b1)==len(b2)):
    raise RuntimeError('冻结数据重复日期或数量不一致')
order=np.argsort(dates.values);dates=dates[order];b1=b1[order];b2=b2[order]
if not(np.isfinite(b1).all() and np.isfinite(b2).all()):raise RuntimeError('输入冻结数据含NaN/Inf')
b1=b1[:,c.DROP_BRANCH1_FIRST:];lat1=lat1[c.DROP_BRANCH1_FIRST:]
keep2=lon2<=c.BRANCH2_MAX_LON+1e-8;lon2=lon2[keep2];b2=b2[:,keep2]
cal=pd.date_range(dates.min(),dates.max(),freq='D');pos=cal.get_indexer(dates)
if (pos<0).any():raise RuntimeError('时间轴不合法')
valid=np.zeros(len(cal),bool);valid[pos]=True
A1=np.full((len(cal),b1.shape[1]),np.nan,np.float32)
A2=np.full((len(cal),b2.shape[1]),np.nan,np.float32)
A1[pos]=b1;A2[pos]=b2
# 缺失值仅以前向可用历史补齐；未来标签永远取 A1/A2 的真实日样本
F1=pd.DataFrame(A1).ffill(limit=c.MAX_INPUT_GAP).to_numpy(dtype=np.float32)
F2=pd.DataFrame(A2).ffill(limit=c.MAX_INPUT_GAP).to_numpy(dtype=np.float32)
H=int(c.HISTORY); L=max(c.LEADS)
missing=(~valid).astype(np.int32);iss=[]
for i in range(H-1,len(cal)-L):
    if not valid[i]:continue
    win=missing[i-H+1:i+1]
    if win.mean()>c.MAX_MISSING_FRAC:continue
    if np.max(np.diff(np.flatnonzero(np.r_[1,win==0,1]))-1)>c.MAX_INPUT_GAP:continue
    if any(not valid[i+d] for d in c.LEADS):continue
    if not(np.isfinite(F1[i-H+1:i+1]).all() and np.isfinite(F2[i-H+1:i+1]).all()):continue
    iss.append(i)
ISS=np.asarray(iss,np.int32)
if not len(ISS):raise RuntimeError('没有可用180→30/60/90起报日期')
if not all(np.all((cal[ISS+d]-cal[ISS]).days==d) for d in c.LEADS):raise RuntimeError('日期错位')
print('Frozen dates',len(dates),'range',dates.min().date(),dates.max().date(),'valid anchors:',len(ISS),'B1dim',b1.shape[1],'B2dim',b2.shape[1])


In [ ]:
# 2｜各年的扩张窗口：训练标签严格早于上一年验证起报，验证标签早于测试起报
FOLDS=[];fr=[];ar=[]
for y in c.TEST_YEARS:
    val_start=pd.Timestamp(int(y)-1,1,1)
    test_start=pd.Timestamp(int(y),1,1)
    next_start=pd.Timestamp(int(y)+1,1,1)
    gap=pd.Timedelta(days=c.PURGE_DAYS)
    train_cut=val_start-gap
    train=ISS[cal[ISS+L]<train_cut]
    val=ISS[(cal[ISS]>=val_start)&(cal[ISS+L]<test_start-gap)]
    test=ISS[(cal[ISS]>=test_start)&(cal[ISS+L]<next_start-gap)]
    if len(train)<c.MIN_TRAIN or len(val)<c.MIN_VAL or len(test)<c.MIN_TEST:
        raise RuntimeError(f'{y} 时序样本不足 train/val/test={(len(train),len(val),len(test))}')
    assert cal[train+L].max() < cal[val].min()-gap
    assert cal[val+L].max() < cal[test].min()-gap
    if not set(cal[train+L]).isdisjoint(cal[val+L]) or not set(cal[val+L]).isdisjoint(cal[test+L]):
        raise RuntimeError(f'{y} 目标日期泄露/相交')
    f={'year':int(y),'train':train,'val':val,'test':test,'train_cut':train_cut,'val_start':val_start,'test_start':test_start}
    FOLDS.append(f)
    fr.append({'test_year':y,'train_n':len(train),'validation_n':len(val),'test_n':len(test),
               'train_last_issue':str(cal[train[-1]].date()),'train_last_target_90d':str(cal[train[-1]+L].date()),
               'validation_first_issue':str(cal[val[0]].date()),'validation_last_target_90d':str(cal[val[-1]+L].date()),
               'test_first_issue':str(cal[test[0]].date()),'test_last_target_90d':str(cal[test[-1]+L].date()),
               'gap_days':c.PURGE_DAYS,'strict_causal':'PASS','seen_in_development':int(y) in (2014,2019)})
    for split,ii in [('train',train),('validation',val),('test',test)]:
        ar.extend({'test_year':y,'split':split,'issue_date':str(cal[i].date()),'target_30':str(cal[i+30].date()),'target_60':str(cal[i+60].date()),'target_90':str(cal[i+90].date())} for i in ii)
fold_table=pd.DataFrame(fr);fold_table.to_csv(TABLES/'01_fold_dates_causal_audit.csv',index=False)
pd.DataFrame(ar).to_csv(TABLES/'02_all_fold_issue_dates_for_future_baselines.csv',index=False)
print('Folds STRICT CAUSAL PASS');display(fold_table)


In [ ]:
# 3｜每折独立在训练日期拟合标准化与99% EOF。训练/验证/测试共享该折拟合结果。
class DailyEOF:
    def fit(self,obs,target=.99):
        obs=np.asarray(obs,np.float64)
        self.mu=np.mean(obs,axis=0);self.sd=np.std(obs,axis=0);self.sd[self.sd<1e-6]=1.
        z=(obs-self.mu)/self.sd
        cov=(z.T@z)/max(1,len(z)-1)
        eig,V=np.linalg.eigh(cov);ii=np.argsort(eig)[::-1]
        eig=np.maximum(eig[ii],0);V=V[:,ii]
        ratios=np.cumsum(eig)/(eig.sum()+1e-12);k=int(np.searchsorted(ratios,target)+1)
        self.V=np.asarray(V[:,:k],np.float32);self.explained=float(ratios[k-1]);self.n=k
        return self
    def z(self,x):return ((np.asarray(x)-self.mu)/self.sd).astype(np.float32)
    def pc(self,x):return (((np.asarray(x)-self.mu)/self.sd)@self.V).astype(np.float32)
    def invert(self,x):return ((np.asarray(x)@self.V.T)*self.sd+self.mu).astype(np.float32)

def make_fold_data(f):
    allowed=(cal<f['train_cut']) & valid
    pp1=DailyEOF().fit(A1[allowed],c.EOF_VARIANCE_TARGET)
    pp2=DailyEOF().fit(A2[allowed],c.EOF_VARIANCE_TARGET)
    if min(pp1.explained,pp2.explained)<c.EOF_VARIANCE_TARGET-1e-6:raise RuntimeError('train-only EOF方差未达标')
    fd=dict(pp1=pp1,pp2=pp2,z1=pp1.z(F1),z2=pp2.z(F2),p1=pp1.pc(F1),p2=pp2.pc(F2))
    snap=EOFSTORE/f"trainonly_EOF_{f['year']}.npz"
    np.savez_compressed(snap,b1_mu=pp1.mu,b1_sd=pp1.sd,b1_V=pp1.V,b2_mu=pp2.mu,b2_sd=pp2.sd,b2_V=pp2.V,
                        lat1=lat1,lon2=lon2,train_cut=str(f['train_cut'].date()),dataset_sha256=DATASET_SHA)
    return fd


In [ ]:
# 3 原架构代码：不更改 GroupNorm 分组数、Dropout、输入特征或层名。
import torch.nn.functional as F
class ChannelSeparatedEncoder(nn.Module):
    """单时刻空间卷积；输入[B,T,2*M] => Conv1d [B*T,2,M]."""
    def __init__(self,feature,branch):
        super().__init__()
        assert feature%2==0, '需要[x, delta]等长拼接输入'
        self.raw_dim=feature//2
        base_c=16 if branch==1 else 32
        k=3 if branch==1 else 5
        channels=base_c*2
        # 正确的对称空间填充：dilation=2 -> padding=k-1，输出仍为M
        self.conv=nn.Conv1d(2,channels,kernel_size=k,dilation=2,padding=k-1)
        self.bn=nn.BatchNorm1d(channels)
        self.fc=nn.Linear(channels*self.raw_dim,32 if branch==1 else 64)
        self.out_dim=self.fc.out_features
        self.drop=nn.Dropout(c.DROPOUT)
    def channelize(self,x):
        B,T,F=x.shape
        if F!=self.raw_dim*2: raise RuntimeError('R53特征维不匹配')
        return torch.stack((x[:,:,:self.raw_dim],x[:,:,self.raw_dim:]),dim=2).reshape(B*T,2,self.raw_dim)
    def forward(self,x):
        B,T,F=x.shape
        z=self.channelize(x)
        z=self.drop(torch.relu(self.bn(self.conv(z))))
        if z.shape[2]!=self.raw_dim:raise RuntimeError('R53 spatial-conv输出宽度发生变化')
        z=self.drop(torch.relu(self.fc(z.flatten(1))))
        return z.reshape(B,T,-1)
class ChannelHybrid(nn.Module):
    def __init__(self,feature,pc_dim,branch,projection=None,use_pc_skip=False):
        super().__init__()
        self.enc=ChannelSeparatedEncoder(feature,branch)
        self.use_pc_skip=bool(use_pc_skip)
        self.raw_dim=feature//2
        self.pc_dim=pc_dim
        if self.use_pc_skip:
            self.register_buffer('V',torch.as_tensor(projection,dtype=torch.float32))
        hidden=32 if branch==1 else 64
        in_dim=self.enc.out_dim+(2*pc_dim if self.use_pc_skip else 0)
        self.rnn=nn.LSTM(in_dim,hidden,batch_first=True)
        self.fc=nn.Linear(hidden,3*pc_dim)
    def eof_skip(self,x):
        if not self.use_pc_skip: raise RuntimeError('no PC skip in this model')
        return torch.cat((x[:,:,:self.raw_dim]@self.V,x[:,:,self.raw_dim:]@self.V),dim=-1)
    def forward(self,x):
        z=self.enc(x)
        if self.use_pc_skip: z=torch.cat((z,self.eof_skip(x)),dim=-1)
        h,_=self.rnn(z)
        return self.fc(h[:,-1]).reshape(len(x),3,self.pc_dim)

def make_r53_model(name,branch,fd):
    if name!='R53-ChannelConv-PCConcat':raise ValueError('只允许原R53-PCConcat编码器')
    feature=fd[f'z{branch}'].shape[1]*2; pdim=fd[f'pp{branch}'].n
    return ChannelHybrid(feature,pdim,branch,fd[f'pp{branch}'].V,use_pc_skip=True)
class JointHiddenEOF(nn.Module):
    def __init__(self,fd,mode='cross'):
        super().__init__()
        if mode not in ('cross','self'): raise ValueError(mode)
        self.mode=mode
        self.net1=make_r53_model('R53-ChannelConv-PCConcat',1,fd)
        self.net2=make_r53_model('R53-ChannelConv-PCConcat',2,fd)
        d1=int(self.net1.fc.in_features);d2=int(self.net2.fc.in_features)
        self.exchange1=nn.Linear(d2 if mode=='cross' else d1,d1)
        self.exchange2=nn.Linear(d1 if mode=='cross' else d2,d2)
        nn.init.zeros_(self.exchange1.weight);nn.init.zeros_(self.exchange1.bias)
        nn.init.zeros_(self.exchange2.weight);nn.init.zeros_(self.exchange2.bias)
    def _hidden(self,net,x):
        a=net.enc(x)
        a=torch.cat([a,net.eof_skip(x)],dim=-1)
        h,_=net.rnn(a)
        return h[:,-1]
    def forward(self,x1,x2):
        h1=self._hidden(self.net1,x1)
        h2=self._hidden(self.net2,x2)
        if self.mode=='cross':
            t1=h1+self.exchange1(torch.tanh(h2))
            t2=h2+self.exchange2(torch.tanh(h1))
        else:
            t1=h1+self.exchange1(torch.tanh(h1))
            t2=h2+self.exchange2(torch.tanh(h2))
        o1=self.net1.fc(t1).reshape(len(x1),3,self.net1.pc_dim)
        o2=self.net2.fc(t2).reshape(len(x2),3,self.net2.pc_dim)
        return o1,o2
class CrossExchangeDropGN(JointHiddenEOF):
    def __init__(self,fd,drop_p):
        super().__init__(fd,'cross')
        self.drop_p=float(drop_p)
        assert 0 <= self.drop_p < 1.0
        for net in (self.net1,self.net2):
            ch=net.enc.conv.out_channels
            assert ch%8==0
            net.enc.bn=nn.GroupNorm(8,ch)

    def forward(self,x1,x2):
        h1=self._hidden(self.net1,x1)
        h2=self._hidden(self.net2,x2)
        m1=self.exchange1(torch.tanh(h2))
        m2=self.exchange2(torch.tanh(h1))
        # 训练期对隐藏特征做独立 dropout；推理时恒等，严禁随机测试增强
        m1=F.dropout(m1,p=self.drop_p,training=self.training)
        m2=F.dropout(m2,p=self.drop_p,training=self.training)
        y1=self.net1.fc(h1+m1).reshape(len(x1),3,self.net1.pc_dim)
        y2=self.net2.fc(h2+m2).reshape(len(x2),3,self.net2.pc_dim)
        return y1,y2

def r65_model(fd,name):
    if name!='R65-CrossDrop60':raise ValueError('仅允许冻结的单模型架构')
    return CrossExchangeDropGN(fd,0.60)

In [ ]:
# 5｜R6.5 原版联合双分支 Dataset + R6.1 物理距离损失；所有参数固定
class JointPairDataset(Dataset):
    def __init__(self,issues,fd):
        self.issues=np.asarray(issues,np.int64)
        self.z1=fd['z1'];self.z2=fd['z2'];self.p1=fd['p1'];self.p2=fd['p2']
    def __len__(self):return len(self.issues)
    def __getitem__(self,j):
        i=int(self.issues[j])
        def pack(z):
            v=z[i-H+1:i+1]
            d=np.empty_like(v);d[0]=0;d[1:]=v[1:]-v[:-1]
            return torch.from_numpy(np.concatenate([v,d],axis=-1).copy())
        y1=np.stack([self.p1[i+d] for d in c.LEADS]);y2=np.stack([self.p2[i+d] for d in c.LEADS])
        return pack(self.z1),pack(self.z2),torch.from_numpy(y1.copy()),torch.from_numpy(y2.copy())

def r61_loss(pred,target,fd,branch):
    # exact original R61 differentiable km-backprojection loss
    pc_mse=F.mse_loss(pred.float(),target.float())
    pp=fd[f'pp{branch}']
    V=torch.as_tensor(pp.V,dtype=torch.float32,device=pred.device)
    sd=torch.as_tensor(pp.sd,dtype=torch.float32,device=pred.device)
    scale=(torch.as_tensor(111.32*np.cos(np.deg2rad(lat1)),dtype=torch.float32,device=pred.device)
           if branch==1 else torch.ones_like(sd)*111.32)
    km=((pred.float()-target.float())@V.T)*((sd*scale)[None,None,:])
    phy=F.smooth_l1_loss(km/c.KM_SCALE,torch.zeros_like(km),beta=c.HUBER_BETA)
    return (1-c.ALPHA)*pc_mse+c.ALPHA*phy

def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

def nparams(m):return sum(int(p.numel()) for p in m.parameters())


In [ ]:
# 6｜后报评分：保留 R6.5 双向最近线段距离定义，同时报告日期及双分支物理坐标误差
# 该指标用于 GLORYS 50m 派生路径后报；不能与 ADT 参考误差直接视为同一真值。
def merge_path(b1coord,b2coord):
    a=np.asarray(b1coord,float);b=np.asarray(b2coord,float)
    candidates=np.flatnonzero(lon2>a[-1])
    k=int(candidates[np.argmin((lon2[candidates]-a[-1])**2+(b[candidates]-lat1[-1])**2)]) if len(candidates) else len(lon2)
    x=np.r_[a,lon2[k:]];y=np.r_[lat1,b[k:]]
    mask=np.isfinite(x)&np.isfinite(y)
    return x[mask],y[mask]

def one_direction(A,B):
    if min(len(A),len(B))<2:return np.asarray([np.nan])
    tree=cKDTree(B);_,kidx=tree.query(A,k=min(3,len(B)))
    kidx=np.asarray(kidx)
    if kidx.ndim==1:kidx=kidx[:,None]
    segments=np.unique(np.clip(np.concatenate([kidx.ravel()-1,kidx.ravel()]),0,len(B)-2))
    v=B[segments+1]-B[segments]
    t=np.clip(np.sum((A[:,None,:]-B[segments][None,:,:])*v[None,:,:],axis=2)/np.maximum(np.sum(v*v,axis=1),1e-12)[None,:],0,1)
    closest=B[segments][None,:,:]+t[:,:,None]*v[None,:,:]
    return np.sqrt(((A[:,None,:]-closest)**2).sum(axis=-1)).min(axis=1)

def errors_for_pair(p1,p2,y1,y2):
    px,py=merge_path(p1,p2);tx,ty=merge_path(y1,y2)
    lat0=float(np.mean(np.r_[py,ty]));lon0=float(np.mean(np.r_[px,tx]))
    cosine=np.cos(np.deg2rad(lat0))
    P=np.column_stack(((px-lon0)*111.32*cosine,(py-lat0)*111.32))
    T=np.column_stack(((tx-lon0)*111.32*cosine,(ty-lat0)*111.32))
    d=np.r_[one_direction(P,T),one_direction(T,P)]
    return float(np.mean(d**2)),float(np.mean(d)),float(np.percentile(d,90))

def score_predictions(f,seed,b1pred,b2pred):
    out=[]
    for j,i in enumerate(f['test']):
        for k,d in enumerate(c.LEADS):
            mse,mae,p90=errors_for_pair(b1pred[j,k],b2pred[j,k],A1[i+d],A2[i+d])
            if not np.isfinite(mse):raise RuntimeError('非有限预测路径评分')
            diff1=(b1pred[j,k]-A1[i+d])*111.32*np.cos(np.deg2rad(lat1))
            diff2=(b2pred[j,k]-A2[i+d])*111.32
            out.append(dict(test_year=int(f['year']),seed=int(seed),model=PAPER_NAME,issue_date=str(cal[i].date()),
                            target_date=str(cal[i+d].date()),lead_days=int(d),
                            path_mse_km2=mse,path_rmse_km=math.sqrt(mse),path_mae_km=mae,path_p90_km=p90,
                            branch1_km_rmse=float(np.sqrt(np.mean(diff1**2))),branch2_km_rmse=float(np.sqrt(np.mean(diff2**2))),
                            previously_development_year=int(f['year']) in (2014,2019)))
    return pd.DataFrame(out)


In [ ]:
# 7｜系统机制资格预检（仅一批，不允许预检失败后自动正式训练）
if DEV.type=='cuda':torch.backends.cudnn.benchmark=False
f=FOLDS[0];fd=make_fold_data(f);seed_all(999)
net=r65_model(fd,INTERNAL_ARCH).to(DEV)
dl=DataLoader(JointPairDataset(f['train'][:2],fd),batch_size=2)
x1,x2,y1,y2=next(iter(dl));x1,x2,y1,y2=[t.to(DEV) for t in (x1,x2,y1,y2)]
net.train();a,b=net(x1,x2)
assert a.shape==(2,3,fd['pp1'].n) and b.shape==(2,3,fd['pp2'].n)
loss=(r61_loss(a,y1,fd,1)+r61_loss(b,y2,fd,2))/2
assert torch.isfinite(loss)
loss.backward()
assert all((p.grad is None or torch.isfinite(p.grad).all()) for p in net.parameters())
net.eval()
with torch.no_grad():
    e1,e2=net(x1,x2);e1b,e2b=net(x1,x2)
assert torch.equal(e1,e1b) and torch.equal(e2,e2b), '推理必须确定性，CrossDrop须关闭'
num=nparams(net)
print('[MECHANISM GATE PASS] parameters',num,'eof_modes',fd['pp1'].n,fd['pp2'].n,'smoke_loss',float(loss.detach().cpu()))
print('Train-only EOF reached:',fd['pp1'].explained,fd['pp2'].explained)
del net,fd,dl,x1,x2,y1,y2,a,b,loss
gc.collect()
if DEV.type=='cuda':torch.cuda.empty_cache()


In [ ]:
# 8｜严格合同的训练/早停/恢复；2014/2019可只在CID签名一致时复用旧冻结权重
# checkpoints 每 epoch 保存 latest.pt；Best永不依据测试集成绩选择。
ARCHITECTURE_ID=hashlib.sha256((INTERNAL_ARCH+'|GN8|crossdrop0.60|pcskip|physical-alpha0.4').encode()).hexdigest()
LEGACY_SEEDS=(71,72)
LEGACY_DEV_YEARS=(2014,2019)
LEGACY_CONFIRM_YEARS=(2006,2011)
LEGACY_PROTOCOL=dict(
    version='R65_GN8_CrossExchangeDrop_v1',dataset_sha256=DATASET_SHA,data_representation='Original frozen',
    history_days=c.HISTORY,lead_days=list(c.LEADS),physical_loss_alpha=.40,
    development_years=list(LEGACY_DEV_YEARS),confirmation_development_years=list(LEGACY_CONFIRM_YEARS),
    seeds=list(LEGACY_SEEDS),epochs=36,patience=8,dropout=.30,cross_exchange_dropout=[.35,.60],
    lr=.001,weight_decay=.0001,reference='same-data same-split R6.4 genuine prediction archive',
    model_choice_from_confirm_years=False,all_QA_years_development_not_blind=True)

def legacy_cid(f,fd,seed):
    old={**LEGACY_PROTOCOL,'QA_year':int(f['year']),'seed':int(seed),'name':INTERNAL_ARCH,
         'tr_end':int(f['train'][-1]),'va_end':int(f['val'][-1]),
         'nPC1':int(fd['pp1'].n),'nPC2':int(fd['pp2'].n)}
    return hashlib.sha256(json.dumps(old,sort_keys=True,ensure_ascii=False).encode()).hexdigest()

def new_contract(f,fd,seed):
    return dict(protocol='R75_expanding_hindcast_v1',arch=ARCHITECTURE_ID,dataset_sha256=DATASET_SHA,
                model_name=PAPER_NAME,year=f['year'],seed=seed,train_end=int(f['train'][-1]),
                val_end=int(f['val'][-1]),test_first=str(cal[f['test'][0]].date()),
                split_gap=c.PURGE_DAYS,eof1=fd['pp1'].n,eof2=fd['pp2'].n,
                epochs=c.EPOCHS,patience=c.PATIENCE,batch=c.BATCH_SIZE,lr=c.LR,wd=c.WEIGHT_DECAY,
                loss_alpha=c.ALPHA,km_scale=c.KM_SCALE,huber=c.HUBER_BETA,history=c.HISTORY,leads=list(c.LEADS),
                p_cross=c.CROSS_DROPOUT,p_encoder=c.DROPOUT,amp=c.USE_AMP,train_only_eof=True)

def cid_from_contract(contract):return hashlib.sha256(json.dumps(contract,sort_keys=True,ensure_ascii=False).encode()).hexdigest()

def restore_completed_or_legacy(f,fd,seed,model,cid,folder):
    best=folder/'best.pt'; status=folder/'status.json'
    if not c.FORCE_TRAIN and best.is_file() and status.is_file():
        ck=torch.load(best,map_location='cpu',weights_only=True)
        if ck.get('cid')!=cid:raise RuntimeError('已存在但合同不一致的 best.pt：'+str(best))
        model.load_state_dict(ck['state'],strict=True)
        meta=json.loads(status.read_text(encoding='utf-8'))
        if meta.get('cid')!=cid:raise RuntimeError('status与best CID不一致')
        print('[R75 REUSE]',f['year'],seed,'frozen fold checkpoint')
        return model,meta
    if c.USE_EXISTING_R65_WEIGHTS and f['year'] in LEGACY_DEV_YEARS and seed in LEGACY_SEEDS and not c.FORCE_TRAIN:
        pt=ROOT/c.SOURCE_R65_WEIGHTS_REL/str(f['year'])/str(seed)/INTERNAL_ARCH/'best.pt'
        if pt.is_file():
            old=torch.load(pt,map_location='cpu',weights_only=True)
            expected=legacy_cid(f,fd,seed)
            if old.get('cid')==expected:
                model.load_state_dict(old['state'],strict=True)
                folder.mkdir(exist_ok=True,parents=True)
                meta=dict(year=f['year'],seed=seed,cid=cid,source='exact_legacy_R65_CID',legacy_checkpoint_sha=sha256_file(pt),
                          best_epoch=int(old.get('epoch',0)),best_val=float(old.get('best_val',np.nan)),elapsed_s=0.0,
                          model_parameters=nparams(model),arch=INTERNAL_ARCH)
                torch.save({'cid':cid,'state':{k:v.detach().cpu() for k,v in model.state_dict().items()},
                            'epoch':meta['best_epoch'],'best_val':meta['best_val'],'legacy_cid':expected},best)
                status.write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding='utf-8')
                print('[R75 EXACT LEGACY REUSE]',f['year'],seed,'sha',meta['legacy_checkpoint_sha'][:12])
                return model,meta
            print('[R75 LEGACY MISMATCH: training from scratch]',f['year'],seed, str(pt))
    return None,None

def run_train(f,fd,seed):
    seed_all(seed)
    m=r65_model(fd,INTERNAL_ARCH).to(DEV)
    folder=WEIGHTS/str(f['year'])/str(seed);folder.mkdir(exist_ok=True,parents=True)
    cid=cid_from_contract(new_contract(f,fd,seed))
    reused,meta=restore_completed_or_legacy(f,fd,seed,m,cid,folder)
    if reused is not None:return reused,meta
    tr=DataLoader(JointPairDataset(f['train'],fd),batch_size=c.BATCH_SIZE,shuffle=True,num_workers=c.NUM_WORKERS)
    va=DataLoader(JointPairDataset(f['val'],fd),batch_size=c.BATCH_SIZE,shuffle=False,num_workers=c.NUM_WORKERS)
    opt=torch.optim.AdamW(m.parameters(),lr=c.LR,weight_decay=c.WEIGHT_DECAY)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,patience=4,factor=.5)
    amp=DEV.type=='cuda' and bool(c.USE_AMP)
    scaler=torch.amp.GradScaler('cuda',enabled=amp)
    first=1;best=float('inf');bad=0;history=[];prior_elapsed=0.0
    latest=folder/'latest.pt';bestfile=folder/'best.pt';status=folder/'status.json'
    if latest.exists() and not c.FORCE_TRAIN:
        state=torch.load(latest,map_location='cpu',weights_only=True)
        if state.get('cid')!=cid:raise RuntimeError('latest.pt 与新协议不一致，需要另选输出目录')
        m.load_state_dict(state['state'],strict=True);opt.load_state_dict(state['opt']);sch.load_state_dict(state['sch'])
        scaler.load_state_dict(state['scaler']);first=int(state['epoch'])+1
        best=float(state['best']);bad=int(state['bad']);history=list(state['history'])
        prior_elapsed=float(state.get('elapsed_s',0.0))
        if 'rng_torch' in state:torch.set_rng_state(state['rng_torch'])
        if DEV.type=='cuda' and 'rng_cuda' in state:torch.cuda.set_rng_state_all(state['rng_cuda'])
        print('[R75 RESUME]',f['year'],seed,'epoch',first)
    epoch_started=time.perf_counter()
    for epoch in range(first,c.EPOCHS+1):
        m.train();tr_sum=0.;tr_n=0
        for x1,x2,y1,y2 in tr:
            x1,x2,y1,y2=[t.to(DEV,non_blocking=True) for t in (x1,x2,y1,y2)]
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda' if amp else 'cpu',enabled=amp):
                a,b=m(x1,x2)
                loss=.5*(r61_loss(a.float(),y1.float(),fd,1)+r61_loss(b.float(),y2.float(),fd,2))
            if not torch.isfinite(loss):raise RuntimeError('训练 loss NaN/Inf')
            scaler.scale(loss).backward();scaler.unscale_(opt);nn.utils.clip_grad_norm_(m.parameters(),1.0)
            scaler.step(opt);scaler.update()
            tr_sum+=float(loss.detach())*len(x1);tr_n+=len(x1)
        m.eval();val_sum=0.;val_n=0
        with torch.inference_mode():
            for x1,x2,y1,y2 in va:
                x1,x2,y1,y2=[t.to(DEV,non_blocking=True) for t in (x1,x2,y1,y2)]
                with torch.autocast(device_type='cuda' if amp else 'cpu',enabled=amp):
                    a,b=m(x1,x2)
                    vloss=.5*(r61_loss(a.float(),y1.float(),fd,1)+r61_loss(b.float(),y2.float(),fd,2))
                val_sum+=float(vloss)*len(x1);val_n+=len(x1)
        value=val_sum/max(1,val_n);sch.step(value)
        if value<best-1e-7:
            best=value;bad=0
            torch.save({'cid':cid,'state':{k:v.detach().cpu() for k,v in m.state_dict().items()},'epoch':epoch,'best_val':best},bestfile)
        else:bad+=1
        history.append(dict(year=f['year'],seed=seed,epoch=epoch,train_loss=tr_sum/max(1,tr_n),val_loss=value,
                            best_val=best,lr=float(opt.param_groups[0]['lr']),bad_epochs=bad,
                            elapsed_s=prior_elapsed+time.perf_counter()-epoch_started))
        pd.DataFrame(history).to_csv(folder/'history.csv',index=False)
        torch.save({'cid':cid,'state':{k:v.detach().cpu() for k,v in m.state_dict().items()},
                    'opt':opt.state_dict(),'sch':sch.state_dict(),'scaler':scaler.state_dict(),
                    'epoch':epoch,'best':best,'bad':bad,'history':history,
                    'elapsed_s':prior_elapsed+time.perf_counter()-epoch_started,
                    'rng_torch':torch.get_rng_state(),'rng_cuda':torch.cuda.get_rng_state_all() if DEV.type=='cuda' else []},latest)
        if epoch==1 or epoch%5==0 or bad>=c.PATIENCE:
            print(f'[R75 TRAIN] {f["year"]} seed {seed} epoch={epoch} train={tr_sum/max(1,tr_n):.5f} val={value:.5f} bad={bad}')
        if bad>=c.PATIENCE:break
    if not bestfile.is_file():raise RuntimeError('无最佳权重，禁止测试')
    ck=torch.load(bestfile,map_location='cpu',weights_only=True)
    if ck['cid']!=cid:raise RuntimeError('best CID 不一致')
    m.load_state_dict(ck['state'],strict=True)
    meta=dict(year=f['year'],seed=seed,cid=cid,source='new_rolling_training',
              best_epoch=int(ck['epoch']),best_val=float(ck['best_val']),model_parameters=nparams(m),
              elapsed_s=prior_elapsed+time.perf_counter()-epoch_started,arch=INTERNAL_ARCH)
    status.write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
    m.eval();return m,meta

def predict_pair(model,fd,issues):
    ld=DataLoader(JointPairDataset(issues,fd),batch_size=c.BATCH_SIZE,shuffle=False,num_workers=c.NUM_WORKERS)
    a=[];b=[];model.eval()
    with torch.inference_mode():
        for x1,x2,_,_ in ld:
            p,q=model(x1.to(DEV),x2.to(DEV))
            a.append(p.float().cpu().numpy());b.append(q.float().cpu().numpy())
    return fd['pp1'].invert(np.concatenate(a)),fd['pp2'].invert(np.concatenate(b))


In [ ]:
# 9｜正式滚动后报：每年严格在过去训练，并在所有预设测试样本上评估。
# 完成单折后立刻保存预测、逐起报误差、训练诊断；中断可续。
if c.ONLY_PREFLIGHT or not c.RUN_FULL:
    print('仅执行 preflight；将 ONLY_PREFLIGHT=False 且 RUN_FULL=True 后才能开始正式后报')
else:
    train_meta=[]
    for fi,f in enumerate(FOLDS,1):
        fold_start=time.perf_counter();print(f'\n========== {fi}/{len(FOLDS)} | Test year {f["year"]} =========')
        fd=make_fold_data(f)
        for seed in c.SEEDS:
            perfile=FORECAST/f"pred_{f['year']}_seed{seed}.npz"
            metricfile=TABLES/f"errors_{f['year']}_seed{seed}.csv"
            folder=WEIGHTS/str(f['year'])/str(seed)
            if (not c.FORCE_TRAIN and metricfile.exists() and perfile.exists() and (folder/'status.json').exists()):
                try:
                    meta=json.loads((folder/'status.json').read_text(encoding='utf-8'))
                    expected=cid_from_contract(new_contract(f,fd,seed))
                    with np.load(perfile,allow_pickle=False) as z:
                        same=(str(z['cid'].item())==expected and len(z['issue_date'])==len(f['test']) and
                              str(z['dataset_sha256'].item())==DATASET_SHA and
                              np.array_equal(z['issue_date'],np.asarray(cal[f['test']].strftime('%Y-%m-%d'),dtype='U10')))
                    if meta.get('cid')==expected and same and len(pd.read_csv(metricfile))==len(f['test'])*len(c.LEADS):
                        print('[R75 SKIP VERIFIED COMPLETE]',f['year'],seed)
                        train_meta.append(meta);continue
                except Exception as e:print('[R75 CACHE NOT REUSED]',repr(e))
            net,meta=run_train(f,fd,seed)
            if net is None:raise RuntimeError('训练未返回模型')
            model_time=time.perf_counter()
            p1,p2=predict_pair(net,fd,f['test'])
            inf_s=time.perf_counter()-model_time
            score=score_predictions(f,seed,p1,p2)
            snapcid=cid_from_contract(new_contract(f,fd,seed))
            np.savez_compressed(perfile,issue_date=np.asarray(cal[f['test']].strftime('%Y-%m-%d'),dtype='U10'),
                                target_date=np.asarray(np.column_stack([cal[f['test']+d].strftime('%Y-%m-%d') for d in c.LEADS]),dtype='U10'),
                                lead_days=np.asarray(c.LEADS),pred_b1_lon=p1,pred_b2_lat=p2,
                                branch1_fixed_lats=lat1,branch2_fixed_lons=lon2,
                                cid=snapcid,dataset_sha256=DATASET_SHA,model=PAPER_NAME)
            score.to_csv(metricfile,index=False)
            meta['test_inference_s']=float(inf_s);meta['n_test_issues']=int(len(f['test']))
            meta['n_lead_forecasts']=int(len(score));meta['preprocessing_sha256']=sha256_file(EOFSTORE/f"trainonly_EOF_{f['year']}.npz")
            (folder/'status.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
            train_meta.append(meta)
            print('[R75 TEST]',f['year'],seed, 'issues',len(f['test']),
                  'L30/60/90 path RMSE (pooled)',
                  score.groupby('lead_days')['path_mse_km2'].mean().pow(.5).round(2).to_dict(),
                  'inf_s',round(inf_s,2))
            del net,p1,p2,score
            if DEV.type=='cuda':torch.cuda.empty_cache()
        del fd;gc.collect()
        pd.DataFrame(train_meta).to_csv(TABLES/'03_run_status.csv',index=False)
        print(f'[R75 FOLD COMPLETE] {f["year"]} wall {time.perf_counter()-fold_start:.1f}s')


In [ ]:
# 10｜统计：年内先聚合不同种子和日期，年间等权；另提供样本量加权聚合与不确定性
metric_parts=sorted(TABLES.glob('errors_*_seed*.csv'))
all_errors=pd.concat([pd.read_csv(q) for q in metric_parts],ignore_index=True) if metric_parts else pd.DataFrame()
if len(all_errors):
    all_errors=all_errors[all_errors['test_year'].isin(c.TEST_YEARS)&all_errors['seed'].isin(c.SEEDS)].copy()
    all_errors.to_csv(TABLES/'04_all_individual_forecast_errors.csv',index=False)
    per_year=[]
    for (year,lead),df in all_errors.groupby(['test_year','lead_days']):
        per_year.append(dict(test_year=int(year),lead_days=int(lead),n_issue=int(df['issue_date'].nunique()),
                             n_seed=int(df['seed'].nunique()),
                             pooled_rmse_km=float(np.sqrt(df['path_mse_km2'].mean())),
                             average_daily_rmse_km=float(df['path_rmse_km'].mean()),
                             average_daily_mae_km=float(df['path_mae_km'].mean()),
                             median_daily_rmse_km=float(df['path_rmse_km'].median()),
                             p90_daily_rmse_km=float(df['path_rmse_km'].quantile(.9)),
                             development_year=int(year) in LEGACY_DEV_YEARS))
    yearly=pd.DataFrame(per_year).sort_values(['test_year','lead_days'])
    yearly.to_csv(TABLES/'05_rolling_year_by_lead.csv',index=False)
    mac=[]
    for lead,grp in yearly.groupby('lead_days'):
        mean_mse=np.mean(np.square(grp['pooled_rmse_km']))
        mac.append(dict(model=PAPER_NAME,lead_days=int(lead),years=len(grp),
                        equal_year_weight_rmse_km=float(np.sqrt(mean_mse)),
                        arithmetic_mean_year_rmse_km=float(grp['pooled_rmse_km'].mean()),
                        year_rmse_sd=float(grp['pooled_rmse_km'].std(ddof=1)) if len(grp)>1 else np.nan,
                        n_total_issues=int(grp['n_issue'].sum()),
                        years_included=';'.join(map(str,sorted(grp['test_year'].astype(int).tolist())))))
    macro=pd.DataFrame(mac);macro.to_csv(TABLES/'06_multi_year_macro_metrics.csv',index=False)
    # year cluster bootstrap, first group by year/issue date to prevent counting seeds as independent dates
    daily=all_errors.groupby(['test_year','lead_days','issue_date'],as_index=False)['path_mse_km2'].mean()
    rng=np.random.default_rng(20261016)
    boots=[]
    for lead,sub in daily.groupby('lead_days'):
        arrays=[q.sort_values('issue_date')['path_mse_km2'].to_numpy(float) for y,q in sub.groupby('test_year')]
        if len(arrays)<2:continue
        scores=[]
        for _ in range(c.BOOT_REPS):
            chosen=rng.integers(0,len(arrays),len(arrays))
            yrm=[]
            for pick in chosen:
                v=arrays[pick];block=min(c.BOOT_BLOCK_DAYS,len(v))
                inds=[]
                while len(inds)<len(v):
                    start=int(rng.integers(0,len(v)))
                    inds.extend((start+np.arange(block))%len(v))
                yrm.append(float(v[np.asarray(inds[:len(v)],int)].mean()))
            scores.append(float(np.sqrt(np.mean(yrm))))
        boots.append(dict(lead_days=int(lead),CI95_lower=float(np.quantile(scores,.025)),
                          CI95_upper=float(np.quantile(scores,.975)),B=c.BOOT_REPS,
                          cluster='test_year and circular 30-day date blocks'))
    ci=pd.DataFrame(boots);ci.to_csv(TABLES/'07_bootstrap_year_date_CI95.csv',index=False)
    print('YEAR MACRO');display(macro)
    if len(ci):display(ci)
    # plot performance of every year, show failures rather than filtering
    col={30:'#437b9b',60:'#b68b5d',90:'#6f8f88'}
    fig,ax=plt.subplots(figsize=(12,5.3))
    for lead,g in yearly.groupby('lead_days'):
        ax.plot(g['test_year'],g['pooled_rmse_km'],marker='o',markersize=4,lw=1.5,color=col.get(int(lead),'gray'),label=f'+{lead} days')
    ax.set(xlabel='Hindcast test year',ylabel='Path RMSE (km)',title=f'{PAPER_NAME} | expanding-window rolling hindcasts')
    ax.legend(ncol=3);ax.grid(alpha=.22);fig.tight_layout()
    for typ in ('png','pdf'):fig.savefig(FIG/f'Fig01_MultiYear_Hindcast_RMSE.{typ}',dpi=c.FIG_DPI)
    plt.close(fig)
    fig,ax=plt.subplots(figsize=(9,4.6))
    for l in c.LEADS:
        g=yearly[yearly['lead_days']==l]
        if len(g):ax.boxplot([g['pooled_rmse_km'].values],positions=[int(l)],widths=13)
    ax.set(xticks=list(c.LEADS),xlabel='Forecast lead (days)',ylabel='Year-level path RMSE (km)',title=f'{PAPER_NAME} | year-to-year variability')
    ax.grid(axis='y',alpha=.22);fig.tight_layout()
    for typ in ('png','pdf'):fig.savefig(FIG/f'Fig02_Interannual_Variability.{typ}',dpi=c.FIG_DPI)
    plt.close(fig)
else:
    print('No completed rolling hindcasts found. Run training cell before this summary.')


In [ ]:
# 11｜自动结果包 SHA256；对已完成、部分完成、未开始状态均写明
status_files=sorted(WEIGHTS.glob('*/*/status.json'))
statuses=[]
for f in status_files:
    try:statuses.append(json.loads(f.read_text(encoding='utf-8')))
    except Exception as e:print('bad status',f,e)
completed={(int(r['year']),int(r['seed'])) for r in statuses if (FORECAST/f"pred_{r['year']}_seed{r['seed']}.npz").exists()}
expected={(int(y),int(s)) for y in c.TEST_YEARS for s in c.SEEDS}
state={'run_utc':datetime.now(timezone.utc).isoformat(),'model':PAPER_NAME,'internal_arch':INTERNAL_ARCH,
       'dataset_sha256':DATASET_SHA,'test_years':list(c.TEST_YEARS),'seeds':list(c.SEEDS),
       'expected_fold_seed_runs':len(expected),'completed_fold_seed_runs':len(completed&expected),
       'complete':expected<=completed,'missing_runs':[{'year':y,'seed':s} for y,s in sorted(expected-completed)],
       'metrics_from_GLORYS_50m_not_ADT':True,
       '2014_and_2019_previously_used_in_development':True,'test_is_not_blind':True,
       'weights_trained_causally_for_each_test_year':True,
       'network_architecture_and_training_contract_fixed':True}
(REPORTS/'R75_completed_status.json').write_text(json.dumps(state,ensure_ascii=False,indent=2),encoding='utf-8')
report="Research protocol: expanding-window; train labels finish 7d before validation issues; validation labels finish 7d before test issues. Each fold EOF is fitted only on preceding training dates. The paper model is called hybrid deep-learning prediction, with internal provenance R65-CrossDrop60. Inputs: 180-day GLORYS-derived pathways; targets: +30/+60/+90 days. BN replaced by GN8, cross-exchange dropout p=0.60 applies only in training. Model hyperparameters frozen; per-year weights retrained under causality. 2014 and 2019 are development-exposed and not blind. ADT reference validation is reported separately in R7.4. Models' relative skill vs Persistence/Climatology/AR-EOF/parameter-matched LSTM will be tested on exactly the same R7.5 issue dates in the next experiment. Full predictions/checkpoints remain local; only result summaries included in ZIP by default."
(REPORTS/'R75_paper_methods_note.txt').write_text(report,encoding='utf-8')
# Zip is written atomically; output does not include huge checkpoints by default
pack=Path(str(ARCHIVE)+'.tmp')
with zipfile.ZipFile(pack,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6,allowZip64=True) as z:
    for folder in (TABLES,REPORTS,FIG):
        for f in folder.rglob('*'):
            if f.is_file():z.write(f,arcname=str(f.relative_to(RUN)))
    for f in sorted(EOFSTORE.glob('*.npz')):
        z.write(f,arcname=str(f.relative_to(RUN)))
    # only sampled prediction files unless explicit huge archive
    files=sorted(FORECAST.glob('pred_*.npz'))
    if not c.INCLUDE_PREDICTIONS_IN_ZIP:files=files[:c.ZIP_PREDICTION_EXAMPLES]
    for f in files:z.write(f,arcname=str(f.relative_to(RUN)))
    if c.INCLUDE_WEIGHTS_IN_ZIP:
        for f in WEIGHTS.rglob('best.pt'):z.write(f,arcname=str(f.relative_to(RUN)))
pack.replace(ARCHIVE)
sha=sha256_file(ARCHIVE)
Path(str(ARCHIVE)+'.sha256').write_text(f'{sha}  {ARCHIVE.name}\n',encoding='utf-8')
with zipfile.ZipFile(ARCHIVE) as z:assert z.testzip() is None
print('R75 ARCHIVE',ARCHIVE,'bytes',ARCHIVE.stat().st_size,'SHA256',sha)
print('Runs',len(completed&expected),'/',len(expected),'complete=',state['complete'])
try:display(FileLink(str(ARCHIVE)));display(FileLink(str(ARCHIVE)+'.sha256'))
except Exception:pass
